# Notebook 2: Data Preprocessing

**Purpose:** This notebook loads, validates, and prepares the LUDB (Lobachevsky University Database) ECG dataset
for model training. It covers:
- Loading raw `.hea` and `.dat` files using the `wfdb` library
- Extracting P-wave annotations (onset, peak, offset)
- Splitting data into Train / Validation / Test sets
- Applying per-lead Z-score normalization
- Applying research-grade augmentations during training

**Research Note:** Only real human ECG records from LUDB are used here. No synthetic data is involved.

## Step 1: Configure Paths and Add Project to Python Path

In [ ]:
import sys
import os
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

# Set project root so we can import from src/
PROJECT_ROOT = str(Path(os.getcwd()).parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Path to the LUDB data
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw', 'ludb')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")

## Step 2: Load the LUDB Dataset

The `LUDBLoader` class reads all 200 ECG records from LUDB.
It returns:
- `signals`: A numpy array of shape `[N, 12, 5000]` — N records, 12 leads, 5000 time steps (10 seconds at 500Hz)
- `annotations`: A list of dicts where each dict contains the P-wave (onset, peak, offset) positions

In [ ]:
from src.data_pipeline.ludb_loader import LUDBLoader

loader = LUDBLoader(DATA_DIR)
signals, annotations = loader.get_all_data()

print(f"Total records loaded: {len(signals)}")
print(f"Signal shape (first record): {signals[0].shape}   [12 leads x 5000 samples]")
print(f"Example P-wave for record 0: {annotations[0]['p_waves'][:3]}")

## Step 3: Split Data into Train, Validation, and Test Sets

We use a fixed seed to guarantee reproducibility.
The split is: **70% Train / 15% Validation / 15% Test**.
The test indices are saved so evaluation always uses the same held-out records.

In [ ]:
np.random.seed(42)

indices = np.arange(len(signals))
np.random.shuffle(indices)

tr_stop  = int(0.70 * len(indices))
val_stop = int(0.85 * len(indices))

idx_tr   = indices[:tr_stop]
idx_val  = indices[tr_stop:val_stop]
idx_test = indices[val_stop:]

# Save test indices so evaluation notebook uses the same held-out records
np.save(os.path.join(PROCESSED_DIR, 'test_indices.npy'), idx_test)
np.save(os.path.join(PROCESSED_DIR, 'val_indices.npy'), idx_val)

print(f"Train records  : {len(idx_tr)}")
print(f"Validation records: {len(idx_val)}")
print(f"Test records   : {len(idx_test)}")
print("Test indices saved to data/processed/test_indices.npy")

## Step 4: Create Dataset Objects

`AtrionInstanceDataset` wraps the raw signals and applies:
- **Per-lead Z-score normalization** (each lead independently)
- **Augmentations** (only during training): time-shift, amplitude scaling, Gaussian noise, lead dropout, and MixUp overlay

In [ ]:
from src.data_pipeline.instance_dataset import AtrionInstanceDataset

train_ds = AtrionInstanceDataset(
    signals[idx_tr],
    [annotations[i] for i in idx_tr],
    is_train=True   # Augmentations ON during training
)
val_ds = AtrionInstanceDataset(
    signals[idx_val],
    [annotations[i] for i in idx_val],
    is_train=False  # No augmentation during evaluation
)
test_ds = AtrionInstanceDataset(
    signals[idx_test],
    [annotations[i] for i in idx_test],
    is_train=False
)

print(f"Train dataset size : {len(train_ds)}")
print(f"Val dataset size   : {len(val_ds)}")
print(f"Test dataset size  : {len(test_ds)}")

## Step 5: Create DataLoaders

DataLoaders batch the records for efficient GPU processing during training.

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=1)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test samples: {len(test_loader)}")

## Step 6: Inspect One Sample

This cell shows the shape of the data items that the model will receive during training.

In [ ]:
sample = train_ds[0]

for key, val in sample.items():
    print(f"{key:10s}: shape={val.shape}, dtype={val.dtype}")

## Step 7: Visualise a Raw ECG Record with P-Wave Labels

This plot shows Lead II from one record with the P-wave regions highlighted in red.
This confirms the annotation loading is correct.

In [ ]:
import matplotlib.pyplot as plt

# Pick the first training record
rec_idx = idx_tr[0]
signal_raw = signals[rec_idx]   # [12, 5000]
ann = annotations[rec_idx]

lead_ii = signal_raw[1]  # Lead II is index 1
time_axis = np.arange(len(lead_ii)) / 500.0  # Convert samples to seconds

plt.figure(figsize=(15, 4))
plt.plot(time_axis, lead_ii, color='steelblue', linewidth=0.8, label='Lead II')

# Highlight P-wave regions
for onset, peak, offset in ann['p_waves']:
    plt.axvspan(onset/500, offset/500, color='red', alpha=0.3, label='P-wave region')
    plt.axvline(peak/500, color='darkred', linewidth=0.7, linestyle='--')

plt.title(f'LUDB Record {rec_idx} - Lead II with P-Wave Annotations')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude (mV)')
# Avoid duplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.tight_layout()
plt.show()

print(f"Number of P-waves in this record: {len(ann['p_waves'])}")

---
**Preprocessing complete. Proceed to `03_model_building/model_architecture.ipynb`.**